# Context-GNN Training Notebook

**Run all 4 cells in order.** This notebook:
1. Mounts Drive & sets up environment
2. Runs preprocessing (creates `processed/` folder if missing)
3. Generates text embeddings (skips if already on Drive)
4. Trains Context-GNN and saves checkpoint

**Requirements:** Enable GPU in Runtime → Change runtime type → T4 GPU

In [ ]:
# ============================================================
# CELL 1: Setup — Mount Drive, Install Dependencies
# ============================================================

import os
from google.colab import drive

if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted')
else:
    drive.mount('/content/drive')

!pip install -q sentence-transformers huggingface_hub pyarrow

import torch

DRIVE_RAW       = '/content/drive/MyDrive/two_tower_data/raw'
DRIVE_PROCESSED = '/content/drive/MyDrive/two_tower_data/processed'
DRIVE_CKPT      = '/content/drive/MyDrive/two_tower_data/checkpoints'
os.makedirs(DRIVE_RAW, exist_ok=True)
os.makedirs(DRIVE_PROCESSED, exist_ok=True)
os.makedirs(DRIVE_CKPT, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected. Enable GPU in Runtime > Change runtime type.')

# Check what already exists
has_raw = os.path.exists(f'{DRIVE_RAW}/amazon_reviews.parquet')
has_processed = os.path.exists(f'{DRIVE_PROCESSED}/stats.json')
has_text_embs = os.path.exists(f'{DRIVE_PROCESSED}/item_text_embeddings.npy')
has_ckpt = os.path.exists(f'{DRIVE_CKPT}/context_gnn_best.pt')

print(f'\nDrive status:')
print(f'  Raw data:        {"YES" if has_raw else "MISSING — Cell 2 will download"}')
print(f'  Processed data:  {"YES" if has_processed else "MISSING — Cell 2 will create"}')
print(f'  Text embeddings: {"YES" if has_text_embs else "MISSING — Cell 3 will create"}')
print(f'  Context-GNN ckpt:{"YES (will retrain)" if has_ckpt else "Not yet"}')
print(f'\n✅ Setup complete')

In [ ]:
# ============================================================
# CELL 2: Download Raw Data + Preprocessing Pipeline
# K-core filtering → feature engineering → train/val/test splits
# Skips download if raw data already on Drive
# Skips preprocessing if processed data already on Drive
# ============================================================

import os, json, re
import pandas as pd
import numpy as np
from collections import defaultdict

DRIVE_RAW       = '/content/drive/MyDrive/two_tower_data/raw'
DRIVE_PROCESSED = '/content/drive/MyDrive/two_tower_data/processed'

# ── Download raw data if needed ──────────────────────────
reviews_path = f'{DRIVE_RAW}/amazon_reviews.parquet'
meta_path    = f'{DRIVE_RAW}/amazon_metadata.parquet'

if os.path.exists(reviews_path) and os.path.exists(meta_path):
    print('✅ Raw data already on Drive — skipping download')
else:
    from huggingface_hub import hf_hub_download
    import shutil

    print('Downloading reviews jsonl (5-10 min)...')
    reviews_file = hf_hub_download(
        repo_id='McAuley-Lab/Amazon-Reviews-2023',
        filename='raw/review_categories/Video_Games.jsonl',
        repo_type='dataset',
        local_dir='/content/tmp_hf'
    )
    reviews_df = pd.read_json(reviews_file, lines=True)
    reviews_df.to_parquet(reviews_path, index=False)
    print(f'  ✅ Reviews: {len(reviews_df):,} rows')

    print('Downloading metadata jsonl...')
    meta_file = hf_hub_download(
        repo_id='McAuley-Lab/Amazon-Reviews-2023',
        filename='raw/meta_categories/meta_Video_Games.jsonl',
        repo_type='dataset',
        local_dir='/content/tmp_hf'
    )
    meta_df = pd.read_json(meta_file, lines=True)
    for col in meta_df.columns:
        if meta_df[col].dtype == 'object':
            meta_df[col] = meta_df[col].astype(str)
    meta_df.to_parquet(meta_path, index=False)
    print(f'  ✅ Metadata: {len(meta_df):,} rows')
    shutil.rmtree('/content/tmp_hf', ignore_errors=True)

# ── Preprocessing (skip if already done) ─────────────────
if os.path.exists(f'{DRIVE_PROCESSED}/stats.json'):
    print('\n✅ Processed data already on Drive — skipping preprocessing')
    with open(f'{DRIVE_PROCESSED}/stats.json') as f:
        stats = json.load(f)
    for k, v in stats.items():
        print(f'  {k}: {v:,.4f}' if isinstance(v, float) else f'  {k}: {v:,}')
else:
    print('\nRunning preprocessing pipeline...')
    reviews = pd.read_parquet(reviews_path)
    meta    = pd.read_parquet(meta_path)
    print(f'  Reviews:  {len(reviews):,}')
    print(f'  Metadata: {len(meta):,}')

    # K-Core Filtering (k=5)
    print('\n  Step 1: K-Core filtering (k=5)...')
    def k_core_filter(df, user_col, item_col, k=5):
        while True:
            item_counts = df[item_col].value_counts()
            df = df[df[item_col].isin(item_counts[item_counts >= k].index)]
            user_counts = df[user_col].value_counts()
            df = df[df[user_col].isin(user_counts[user_counts >= k].index)]
            if df[item_col].value_counts().min() >= k and df[user_col].value_counts().min() >= k:
                break
        return df.reset_index(drop=True)

    reviews = reviews.rename(columns={'parent_asin': 'item_id'})
    filtered = k_core_filter(reviews, 'user_id', 'item_id', k=5)
    print(f'  Before: {len(reviews):,} → After: {len(filtered):,}')

    # ID Mappings
    print('  Step 2: Creating ID mappings...')
    users = sorted(filtered['user_id'].unique())
    items = sorted(filtered['item_id'].unique())
    user2idx = {u: i for i, u in enumerate(users)}
    item2idx = {it: i for i, it in enumerate(items)}
    idx2user = {i: u for u, i in user2idx.items()}
    idx2item = {i: it for it, i in item2idx.items()}
    filtered['user_idx'] = filtered['user_id'].map(user2idx)
    filtered['item_idx'] = filtered['item_id'].map(item2idx)
    N_USERS = len(users)
    N_ITEMS = len(items)
    print(f'  Users: {N_USERS:,}  |  Items: {N_ITEMS:,}')

    # Train/Val/Test Split (leave-last-2-out)
    print('  Step 3: Train/Val/Test split (leave-last-2-out)...')
    filtered = filtered.sort_values(['user_idx', 'timestamp']).reset_index(drop=True)
    train_rows, val_rows, test_rows = [], [], []
    for user_idx, group in filtered.groupby('user_idx'):
        rows = group.to_dict('records')
        if len(rows) < 3:
            train_rows.extend(rows)
        else:
            train_rows.extend(rows[:-2])
            val_rows.append(rows[-2])
            test_rows.append(rows[-1])
    train_df = pd.DataFrame(train_rows).reset_index(drop=True)
    val_df   = pd.DataFrame(val_rows).reset_index(drop=True)
    test_df  = pd.DataFrame(test_rows).reset_index(drop=True)
    print(f'  Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

    # User Features
    print('  Step 4: Engineering user features...')
    user_feats = train_df.groupby('user_idx').agg(
        n_purchases     = ('item_idx', 'count'),
        avg_rating      = ('rating', 'mean'),
        rating_std      = ('rating', 'std'),
        total_helpful   = ('helpful_vote', 'sum'),
        verified_ratio  = ('verified_purchase', 'mean'),
    ).reset_index()
    user_feats['rating_std'] = user_feats['rating_std'].fillna(0)
    global_avg = train_df['rating'].mean()
    user_feats['rating_bias']      = user_feats['avg_rating'] - global_avg
    user_feats['log_n_purchases']  = np.log1p(user_feats['n_purchases'])
    user_feats['log_helpful']      = np.log1p(user_feats['total_helpful'])
    all_user_df = pd.DataFrame({'user_idx': range(N_USERS)})
    user_feats  = all_user_df.merge(user_feats, on='user_idx', how='left').fillna(0)
    user_feats  = user_feats.sort_values('user_idx').reset_index(drop=True)
    feat_cols = [c for c in user_feats.columns if c != 'user_idx']
    for col in feat_cols:
        col_max = user_feats[col].max()
        if col_max > 0:
            user_feats[col] = user_feats[col] / col_max
    print(f'  User features: {len(feat_cols)} dims')

    # Item Features
    print('  Step 5: Engineering item features...')
    item_stats = train_df.groupby('item_idx').agg(
        n_reviews      = ('user_idx', 'count'),
        avg_rating     = ('rating', 'mean'),
        rating_std     = ('rating', 'std'),
        total_helpful  = ('helpful_vote', 'sum'),
        verified_ratio = ('verified_purchase', 'mean'),
        frac_5star     = ('rating', lambda x: (x==5).mean()),
        frac_4star     = ('rating', lambda x: (x==4).mean()),
        frac_3star     = ('rating', lambda x: (x==3).mean()),
        frac_2star     = ('rating', lambda x: (x==2).mean()),
        frac_1star     = ('rating', lambda x: (x==1).mean()),
    ).reset_index()
    item_stats['rating_std'] = item_stats['rating_std'].fillna(0)
    meta_slim = meta[['parent_asin','price','average_rating']].copy()
    meta_slim.columns = ['item_id','meta_price','meta_avg_rating']
    def parse_price(p):
        if pd.isna(p) or p == 'None' or p == 'nan':
            return np.nan
        nums = re.findall(r'\d+\.?\d*', str(p))
        return float(nums[0]) if nums else np.nan
    meta_slim['meta_price'] = meta_slim['meta_price'].apply(parse_price)
    meta_slim['meta_avg_rating'] = pd.to_numeric(meta_slim['meta_avg_rating'], errors='coerce')
    meta_slim['item_idx'] = meta_slim['item_id'].map(item2idx)
    meta_slim = meta_slim.dropna(subset=['item_idx'])
    meta_slim['item_idx'] = meta_slim['item_idx'].astype(int)
    item_stats = item_stats.merge(meta_slim[['item_idx','meta_price','meta_avg_rating']],
                                   on='item_idx', how='left')
    item_stats['log_n_reviews']   = np.log1p(item_stats['n_reviews'])
    item_stats['popularity_pct']  = item_stats['n_reviews'].rank(pct=True)
    item_stats['log_price']       = np.log1p(item_stats['meta_price'].fillna(0))
    item_stats = item_stats.fillna(0)
    all_item_df = pd.DataFrame({'item_idx': range(N_ITEMS)})
    item_feats  = all_item_df.merge(item_stats, on='item_idx', how='left').fillna(0)
    item_feats  = item_feats.sort_values('item_idx').reset_index(drop=True)
    feat_cols_i = [c for c in item_feats.columns if c != 'item_idx']
    for col in feat_cols_i:
        col_max = item_feats[col].max()
        if col_max > 0:
            item_feats[col] = item_feats[col] / col_max
    print(f'  Item features: {len(feat_cols_i)} dims')

    # Save everything
    print('  Step 6: Saving to Drive...')
    train_df.to_parquet(f'{DRIVE_PROCESSED}/train.parquet', index=False)
    val_df.to_parquet(f'{DRIVE_PROCESSED}/val.parquet', index=False)
    test_df.to_parquet(f'{DRIVE_PROCESSED}/test.parquet', index=False)
    user_feats.drop(columns=['user_idx']).to_parquet(
        f'{DRIVE_PROCESSED}/user_features.parquet', index=False)
    item_feats.drop(columns=['item_idx']).to_parquet(
        f'{DRIVE_PROCESSED}/item_features.parquet', index=False)
    with open(f'{DRIVE_PROCESSED}/user2idx.json', 'w') as f:
        json.dump(user2idx, f)
    with open(f'{DRIVE_PROCESSED}/item2idx.json', 'w') as f:
        json.dump(item2idx, f)
    with open(f'{DRIVE_PROCESSED}/idx2user.json', 'w') as f:
        json.dump(idx2user, f)
    with open(f'{DRIVE_PROCESSED}/idx2item.json', 'w') as f:
        json.dump(idx2item, f)
    stats = {
        'n_users': N_USERS, 'n_items': N_ITEMS,
        'n_train': len(train_df), 'n_val': len(val_df), 'n_test': len(test_df),
        'n_user_features': len(feat_cols),
        'n_item_features': len(feat_cols_i),
        'sparsity': 1 - len(filtered) / (N_USERS * N_ITEMS)
    }
    with open(f'{DRIVE_PROCESSED}/stats.json', 'w') as f:
        json.dump(stats, f, indent=2)

    print(f'\n  PREPROCESSING COMPLETE')
    for k, v in stats.items():
        print(f'  {k}: {v:,.4f}' if isinstance(v, float) else f'  {k}: {v:,}')

print(f'\n✅ Processed data ready on Drive')

In [ ]:
# ============================================================
# CELL 3: Generate Item Text Embeddings
# Skips if already on Drive
# ============================================================

import os, json
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

DRIVE_RAW       = '/content/drive/MyDrive/two_tower_data/raw'
DRIVE_PROCESSED = '/content/drive/MyDrive/two_tower_data/processed'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

emb_path = f'{DRIVE_PROCESSED}/item_text_embeddings.npy'

if os.path.exists(emb_path):
    print('✅ Text embeddings already on Drive — skipping')
    text_embs = np.load(emb_path)
    print(f'   Shape: {text_embs.shape}')
else:
    meta_df = pd.read_parquet(f'{DRIVE_RAW}/amazon_metadata.parquet')
    with open(f'{DRIVE_PROCESSED}/item2idx.json') as f:
        item2idx = json.load(f)

    N_ITEMS = len(item2idx)
    print(f'Items: {N_ITEMS:,}')

    meta_df['title'] = meta_df['title'].fillna('Unknown Product')
    title_lookup = dict(zip(meta_df['parent_asin'], meta_df['title']))

    titles = []
    missing = 0
    for asin, idx in sorted(item2idx.items(), key=lambda x: x[1]):
        t = title_lookup.get(asin, 'Unknown Product')
        if t == 'nan' or t == 'None':
            t = 'Unknown Product'
            missing += 1
        titles.append(t)

    print(f'Titles mapped: {len(titles):,} | Missing: {missing}')

    print(f'\nEncoding {len(titles):,} titles with all-MiniLM-L6-v2...')
    model = SentenceTransformer('all-MiniLM-L6-v2', device=str(device))
    text_embs = model.encode(
        titles,
        batch_size=512,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    np.save(emb_path, text_embs)
    print(f'\n✅ Text embeddings saved ({text_embs.shape})')

print(f'Text embedding dim: {text_embs.shape[1]}')

In [ ]:
# ============================================================
# CELL 4: Context-GNN Training
# LightGCN-style propagation + feature projection + learnable gate
# Memory-efficient: no per-edge attention (fits T4 16GB)
# ============================================================

import os, json, gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader

DRIVE_PROCESSED = '/content/drive/MyDrive/two_tower_data/processed'
DRIVE_CKPT      = '/content/drive/MyDrive/two_tower_data/checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Load data ────────────────────────────────────────────
with open(f'{DRIVE_PROCESSED}/stats.json') as f:
    stats = json.load(f)
N_USERS      = stats['n_users']
N_ITEMS      = stats['n_items']
N_USER_FEATS = stats['n_user_features']
N_ITEM_FEATS = stats['n_item_features']

train_df     = pd.read_parquet(f'{DRIVE_PROCESSED}/train.parquet')
val_df       = pd.read_parquet(f'{DRIVE_PROCESSED}/val.parquet')
user_feats   = pd.read_parquet(f'{DRIVE_PROCESSED}/user_features.parquet').values.astype(np.float32)
item_feats   = pd.read_parquet(f'{DRIVE_PROCESSED}/item_features.parquet').values.astype(np.float32)
text_embs    = np.load(f'{DRIVE_PROCESSED}/item_text_embeddings.npy').astype(np.float32)
user_history = train_df.groupby('user_idx')['item_idx'].apply(set).to_dict()

user_feats_t = torch.tensor(user_feats, device=device)
item_feats_t = torch.tensor(item_feats, device=device)
text_embs_t  = torch.tensor(text_embs, device=device)
N_TEXT_DIM   = text_embs.shape[1]

print(f'Users: {N_USERS:,} | Items: {N_ITEMS:,}')
print(f'User feats: {N_USER_FEATS} | Item feats: {N_ITEM_FEATS} | Text dim: {N_TEXT_DIM}')

# ── Build adjacency matrix ───────────────────────────────
def build_adj(train_df, n_users, n_items):
    users = train_df['user_idx'].values
    items = train_df['item_idx'].values + n_users
    row   = np.concatenate([users, items])
    col   = np.concatenate([items, users])
    N     = n_users + n_items
    deg   = np.bincount(row, minlength=N).astype(np.float32)
    deg   = np.maximum(deg, 1.0)
    d_inv = 1.0 / np.sqrt(deg)
    vals  = d_inv[row] * d_inv[col]
    idx   = torch.tensor(np.stack([row, col]), dtype=torch.long)
    v     = torch.tensor(vals, dtype=torch.float32)
    return torch.sparse_coo_tensor(idx, v, (N, N)).coalesce().to(device)

adj = build_adj(train_df, N_USERS, N_ITEMS)
print(f'Adj matrix: {adj.shape[0]:,} x {adj.shape[1]:,} | edges: {adj._nnz():,}')

# ── Context-GNN Model ───────────────────────────────────
class ContextGNN(nn.Module):
    def __init__(self, n_users, n_items, n_user_feats, n_item_feats,
                 text_dim=384, dim=64, n_layers=3, dropout=0.1):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.n_layers = n_layers
        self.dim = dim

        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)

        self.user_feat_proj = nn.Linear(n_user_feats, dim, bias=False)
        self.item_feat_proj = nn.Linear(n_item_feats, dim, bias=False)
        self.text_proj = nn.Linear(text_dim, dim, bias=False)

        self.feat_gate = nn.Parameter(torch.tensor(0.3))
        self.dropout = nn.Dropout(dropout)

    def propagate(self, adj):
        E0 = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)

        embs = [E0]
        E = E0
        for _ in range(self.n_layers):
            E = torch.sparse.mm(adj, E)
            E = self.dropout(E)
            embs.append(E)

        E_graph = torch.stack(embs, dim=1).mean(dim=1)

        u_feat = self.user_feat_proj(user_feats_t)
        i_feat = self.item_feat_proj(item_feats_t) + self.text_proj(text_embs_t)
        node_ctx = torch.cat([u_feat, i_feat], dim=0)

        gate = torch.sigmoid(self.feat_gate)
        E_final = (1 - gate) * E_graph + gate * node_ctx

        return E_final[:self.n_users], E_final[self.n_users:]

    def bpr_loss(self, users, pos_items, neg_items, adj):
        user_embs, item_embs = self.propagate(adj)
        u  = user_embs[users]
        pi = item_embs[pos_items]
        ni = item_embs[neg_items]
        pos_scores = (u * pi).sum(dim=1)
        neg_scores = (u * ni).sum(dim=1)
        loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()
        reg = (self.user_emb(users).norm(2).pow(2) +
               self.item_emb(pos_items).norm(2).pow(2) +
               self.item_emb(neg_items).norm(2).pow(2)) / len(users)
        return loss + 1e-4 * reg

# ── Dataset ──────────────────────────────────────────────
class BPRDataset(Dataset):
    def __init__(self, df, n_items, user_history):
        self.users     = df['user_idx'].values
        self.pos_items = df['item_idx'].values
        self.n_items   = n_items
        self.user_history = user_history
    def __len__(self): return len(self.users)
    def __getitem__(self, idx):
        u   = self.users[idx]
        pos = self.pos_items[idx]
        while True:
            neg = np.random.randint(0, self.n_items)
            if neg not in self.user_history.get(u, set()):
                break
        return u, pos, neg

# ── Evaluation ───────────────────────────────────────────
def evaluate(model, adj, val_df, user_history, n_items, K=10, n_neg=100, n_eval=2000):
    model.eval()
    rng = np.random.default_rng(42)
    hr_list, ndcg_list = [], []
    sample_df = val_df.sample(min(n_eval, len(val_df)), random_state=42)
    with torch.no_grad():
        user_embs, item_embs = model.propagate(adj)
        for _, row in sample_df.iterrows():
            u, pos = int(row['user_idx']), int(row['item_idx'])
            seen   = user_history.get(u, set()) | {pos}
            negs   = []
            while len(negs) < n_neg:
                cands = rng.integers(0, n_items, n_neg * 2).tolist()
                negs.extend([c for c in cands if c not in seen])
            negs      = negs[:n_neg]
            candidates = [pos] + negs
            u_emb  = user_embs[u].unsqueeze(0)
            i_embs = item_embs[candidates]
            scores = (u_emb * i_embs).sum(dim=1).cpu().numpy()
            rank   = int(np.where(np.argsort(-scores) == 0)[0][0]) + 1
            hr_list.append(1.0 if rank <= K else 0.0)
            ndcg_list.append(1.0 / np.log2(rank + 1) if rank <= K else 0.0)
    return np.mean(hr_list), np.mean(ndcg_list)

# ── Clear GPU memory before training ────────────────────
gc.collect()
torch.cuda.empty_cache()

# ── Training ─────────────────────────────────────────────
BATCH_SIZE = 2048
EPOCHS     = 30
PATIENCE   = 5
LR         = 1e-3

model     = ContextGNN(N_USERS, N_ITEMS, N_USER_FEATS, N_ITEM_FEATS,
                        text_dim=N_TEXT_DIM, dim=64, n_layers=3, dropout=0.1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
dataset   = BPRDataset(train_df, N_ITEMS, user_history)
loader    = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

best_hr, best_epoch, patience_counter = 0.0, 0, 0

print(f'\nTraining Context-GNN (3 layers, dim=64, feature-gated) for {EPOCHS} epochs...')
print(f'Batch: {BATCH_SIZE} | LR: {LR} | Dropout: 0.1')
if torch.cuda.is_available():
    print(f'GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB used / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB total')
print(f'\n{"Epoch":>5} {"Loss":>10} {"HR@10":>8} {"NDCG@10":>10} {"Gate":>6}')
print('-' * 46)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0

    for users, pos_items, neg_items in loader:
        users     = users.to(device)
        pos_items = pos_items.to(device)
        neg_items = neg_items.to(device)

        loss = model.bpr_loss(users, pos_items, neg_items, adj)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    hr, ndcg = evaluate(model, adj, val_df, user_history, N_ITEMS)
    gate_val = torch.sigmoid(model.feat_gate).item()
    print(f'{epoch:>5} {avg_loss:>10.4f} {hr:>8.4f} {ndcg:>10.4f} {gate_val:>6.2f}')

    if hr > best_hr:
        best_hr, best_epoch, patience_counter = hr, epoch, 0
        torch.save({
            'epoch': epoch, 'hr': hr, 'ndcg': ndcg,
            'gate': gate_val,
            'model_state': model.state_dict(),
            'config': {
                'arch': 'Context-GNN (LightGCN propagation + feature gate)',
                'n_layers': 3, 'dim': 64, 'dropout': 0.1,
                'text_dim': N_TEXT_DIM,
            }
        }, f'{DRIVE_CKPT}/context_gnn_best.pt')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch}')
            break

# ── Results ──────────────────────────────────────────────
ckpt = torch.load(f'{DRIVE_CKPT}/context_gnn_best.pt', weights_only=False)
gate = ckpt['gate']

print(f'\n{"="*55}')
print(f'  CONTEXT-GNN RESULTS')
print(f'{"="*55}')
print(f'  Best Epoch:     {ckpt["epoch"]}')
print(f'  HR@10:          {ckpt["hr"]:.4f}')
print(f'  NDCG@10:        {ckpt["ndcg"]:.4f}')
print(f'  Feature Gate:   {gate:.2f} (graph={1-gate:.0%}, features={gate:.0%})')
print(f'\n  LEADERBOARD:')
print(f'  MF:             HR@10=0.6825')
print(f'  Two-Tower v5:   HR@10=0.6395')
print(f'  LightGCN:       HR@10=0.7290')
print(f'  Context-GNN:    HR@10={ckpt["hr"]:.4f}  <-- NEW')
print(f'\n  Key insight: Feature gate = {gate:.2f}')
print(f'  → Graph signal contributes {1-gate:.0%}, features contribute {gate:.0%}')
print(f'  Compare with FM Two-Tower: ID=63%, GRU=27%, Features=10%')